# Gröbner basis method to solve a polynomial non linear system

In [2]:
import sympy as sp

def solve_groebner_triangular_system(groebner_basis):
    """
    Solves a triangular system coming from a Groebner basis
    
    Args:
        groebner_basis: List of all polinomials, triangular form
    
    Returns:
        Solutions list as a dictionary {var: value}
    """
    if not groebner_basis:
        return [{}]
    
    # Takes the first equation in the triangular system
    current_poly = groebner_basis[0]
    
    # Finds the variable w.r.to wich solve the system, the one of igher degree
    vars_in_poly = current_poly.free_symbols
    if not vars_in_poly:
        return []
    
    # Solves the equation in the first variable
    current_var = next(iter(vars_in_poly))
    solutions = sp.solve(current_poly, current_var)
    solutions = [sol.evalf() if hasattr(sol, 'evalf') else sol for sol in solutions]
    
    # With no solutions, the system is not solvable
    if not solutions:
        return []
    
    # Recursion step: substitutes the values in the following equations
    remaining_basis = groebner_basis[1:]
    all_solutions = []
    
    for sol_val in solutions:
        # Substitutes the variable in the other equations
        substituted_basis = [poly.subs(current_var, sol_val) for poly in remaining_basis]
        
        # Recursively solves the reduced system
        partial_solutions = solve_groebner_triangular_system(substituted_basis)
        
        # Adds the solution to the partial solutions
        for psol in partial_solutions:
            complete_sol = {current_var: sol_val}
            complete_sol.update(psol)
            all_solutions.append(complete_sol)
    
    return all_solutions


def Triangular_Groebner(F,variables,o='grlex'):
    """
    Orders into lexicographic order, and makes the first equation become the one with only one variable, the secondo one depending only on the first and so on
    """
    G = sp.groebner(F, *variables, order=o)
    A = list(G.fglm('lex'))
    B = A[::-1]  # Triangular form
    return B


In [ ]:

x, y, z = sp.symbols('x y z')
Variables = [z, y, x]

# Our system for N_PC = 2
f1 = 15*x**2 + 5*y**2 + 3*z**2 - 15
f2 = 10*x*y + 4*y*z - 1
f3 = 7*y**2 + 3*z**2 + 21*x*z

F = [f1, f2, f3]

# Groebner basis calculation
B = Triangular_Groebner(F,Variables)

print("Triangular Gröbner basis:")
for i in range(len(F)):
    monic_poly = sp.Poly(B[i], *Variables)
    B[i] = monic_poly / monic_poly.LC()
    print(f"B[{i}] = {B[i]}")

# Uses the function defined
solutions = solve_groebner_triangular_system(B)

vettore = []
for i in range(len(solutions)):
    vettore.append(list(solutions[i].values()))

print("\nSolutions found:")
for i in range(len(solutions)):
    print(solutions[i])
    print(f"{vettore[i]}\n")


# Verifies that these are solutions
print("\nVerify that the approximated solutions have a small residual:")
for sol in solutions:
    res = 0
    for eq in F:
        res += abs(eq.subs(sol).evalf())        # calculate the 1-norm of the vector of residuals
    print(f"{sol} gives a residual of \n {res}")

In [ ]:

w, x, y, z = sp.symbols('w x y z')
Variables = [z, y, x, w]

s = sp.Rational(1,5)
mu = 1

# Our system for N_PC = 2
f1 = w**2 + sp.Rational(1,3)*x**2 + sp.Rational(1,5)*y**2 + sp.Rational(1,7)*z**2 - mu
f2 = sp.Rational(2,3)*w*x + sp.Rational(4,15)*x*y + sp.Rational(6,35)*y*z - sp.Rational(1,3)*s
f3 = sp.Rational(1,15)*x**2 + sp.Rational(1,35)*y**2 +sp.Rational(2,105)*z**2 + sp.Rational(1,5)*w*y + sp.Rational(3,35)*x*z
f4 = sp.Rational(3,35)*x*y + sp.Rational(4,105)*y*z + sp.Rational(1,7)*w*z


F = [f1, f2, f3, f4]

# Groebner basis calculation
B = Triangular_Groebner(F,Variables)

print("Triangular Gröbner basis:")
for i in range(len(F)):
    monic_poly = sp.Poly(B[i], *Variables)
    B[i] = monic_poly / monic_poly.LC()
    print(f"B[{i}] = {B[i]}")

# Uses the function defined
solutions = solve_groebner_triangular_system(B)

vettore = []
for i in range(len(solutions)):
    vettore.append([float(val.evalf()) for val in solutions[i].values()])
    
print("\nSolutions found:\n")
for i in range(len(solutions)):
    #print(solutions[i])
    print(f"{vettore[i]}\n")
    

# Verifies that these are solutions
print("\nVerify that the approximated solutions have a small residual:")
for sol in solutions:
    res = 0
    for eq in F:
        res += abs(eq.subs(sol).evalf())        # calculate the 1-norm of the vector of residuals
    print(f"{sol} gives a residual of \n {res}")

In [ ]:
from scipy.optimize import root

# Converti F in una funzione numerica che accetta un array
G = sp.lambdify(Variables, F, "numpy")

# Definisci una wrapper function per root
def func(vals):
    w_val, x_val, y_val, z_val = vals  # Spacchetta l'array
    return G(w_val, x_val, y_val, z_val)  # Restituisce una lista

# Verifica le soluzioni con root
print("\nVerifica con scipy.optimize.root:\n")
for i, guess in enumerate(vettore):
    
    sol_numerica = root(func, guess[::-1])  # Ora funziona!
    print(f"Guess {i+1}:     {guess}")
    print(f"Soluzione {i+1}: {sol_numerica.x[::-1]}")
    print("Residuo:", sol_numerica.fun)
    print("\n")

In [ ]:
x, y, z, mu, s = sp.symbols('x y z mu s')
Variables = [z, y, x]

# Our system for N_PC = 2
f1 = 15*x**2 + 5*y**2 + 3*z**2 - 15*mu
f2 = 10*x*y + 4*y*z - 1*s
f3 = 7*y**2 + 3*z**2 + 21*x*z

F = [f1, f2, f3]

# Groebner basis calculation
B = Triangular_Groebner(F,Variables)
print(B)

In [5]:
import sympy as sp
x, y, z = sp.symbols('x y z')
Variables = [z, y, x]

# Our system for N_PC = 2
f1 = sp.Rational(2,1)*x**2 + sp.Rational(2,3)*y**2 + sp.Rational(2,5)*z**2 - 2*x - sp.Rational(2,15)*y
f2 = sp.Rational(4,3)*x*y+sp.Rational(8,15)*y*z-sp.Rational(2,3)*y-sp.Rational(2,15)*x-sp.Rational(4,75)*z
f3 = sp.Rational(4,15)*y**2+sp.Rational(4,35)*z**2+sp.Rational(4,5)*x*z-sp.Rational(2,5)*z-sp.Rational(4,75)*y

F = [f1, f2, f3]

# Groebner basis calculation
B = Triangular_Groebner(F,Variables)

print("Triangular Gröbner basis:")
for i in range(len(F)):
    monic_poly = sp.Poly(B[i], *Variables)
    B[i] = monic_poly / monic_poly.LC()
    print(f"B[{i}] = {B[i]}")

# Uses the function defined
solutions = solve_groebner_triangular_system(B)

vettore = []
for i in range(len(solutions)):
    vettore.append(list(solutions[i].values()))

print("\nSolutions found:")
for i in range(len(solutions)):
    print(solutions[i])
    print(f"{vettore[i]}\n")


# Verifies that these are solutions
print("\nVerify that the approximated solutions have a small residual:")
for sol in solutions:
    res = 0
    for eq in F:
        res += abs(eq.subs(sol).evalf())        # calculate the 1-norm of the vector of residuals
    print(f"{sol} gives a residual of \n {res}")

Triangular Gröbner basis:
B[0] = x**8 - 4*x**7 + 1187*x**6/180 - 347*x**5/60 + 35237377*x**4/12150000 - 5031127*x**3/6075000 + 489923743*x**2/3936600000 - 5976719*x/787320000
B[1] = 9541983789000*x**7/642308333 - 33396943261500*x**6/642308333 + 429825216150*x**5/5892737 - 33635013247125*x**4/642308333 + 324700019508798*x**3/16057708325 - 63636488853822*x**2/16057708325 + 4908861785234*x/16057708325 + y
B[2] = -3713197950000*x**7/642308333 + 12996192825000*x**6/642308333 - 164771313750*x**5/5892737 + 12409700934375*x**4/642308333 - 8940244567077*x**3/1284616666 + 3174315613731*x**2/2569233332 - 224316922077*x/2569233332 + z

Solutions found:
{x: 0, y: 0, z: 0}
[0, 0, 0]

{x: 1.00000000000000, y: 0.200000000000000, z: 3.63797880709171e-12}
[1.00000000000000, 0.200000000000000, 3.63797880709171e-12]

{x: 0.208865101992724, y: -0.488490404445741, z: 0.515429363673519}
[0.208865101992724, -0.488490404445741, 0.515429363673519]

{x: 0.285395325036392, y: 0.601270130609294, z: 0.7858782312484